<a href="https://colab.research.google.com/github/ShrutikaPatil01/AI_agent/blob/main/AI_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0 langchain-community==0.4.1 langchain-core==1.0.4 requests==2.32.5 duckduckgo-search==8.1.1 ddgs==9.9.0 langsmith==0.4.42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 28.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.9/401.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take i

In [2]:
!pip show langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-core
Version: 1.0

In [3]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
WHETHER_API_KEY=userdata.get('WHETHER_API_KEY')


In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool= DuckDuckGoSearchRun()

In [6]:

@tool
def get_weather_data(city: str) -> str:
  """
  This tool fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={WHETHER_API_KEY}&query={city}' # Enter the weatherstack api key before use

  response = requests.get(url)

  return response.json()

In [7]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_kwargs={"tool_choice":"auto"}
)

In [8]:
from langchain.agents import create_agent
from langsmith import Client

In [9]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [10]:
from langsmith import Client
client=Client()
prompt=client.pull_prompt("hwchase17/react")
prompt_template_string=prompt.template

In [11]:
# Step 3: Create the agent manually with the pulled prompt
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt=prompt_template_string
)

In [12]:
response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Find the capital of Madhya Pradesh"}
    ]}
)

In [13]:
for messages in response['messages']:
  print(messages)

content='Find the capital of Madhya Pradesh' additional_kwargs={} response_metadata={} id='362ed2d0-3816-4c89-a0ee-29c831d5e724'
content='Question: Find the capital of Madhya Pradesh  \nThought: The capital of Madhya Pradesh is Bhopal.  \nFinal Answer: Bhopal' additional_kwargs={'reasoning_content': "We need to answer: capital of Madhya Pradesh. It's Bhopal. No need for tool."} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 315, 'total_tokens': 376, 'completion_time': 0.065513926, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.024804539, 'prompt_tokens_details': None, 'queue_time': 0.040720725, 'total_time': 0.090318465}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--5753028b-9be7-46c9-bd2b-1a7f63eb983a-0' usage_metadata={'input_tokens': 315, 'output_tokens': 61, 'total_tokens': 376}


In [14]:
!pip install -q gradio

In [15]:
import gradio as gr

In [16]:
def get_response(query):
  response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
  )

  return response['messages'][-1].content

In [17]:
iface = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        label="Ask a question to your AI Agent",
        placeholder="e.g., Find the capital of Madhya Pradesh, then find it's current weather condition",
        lines=2
    ),
    outputs=gr.Textbox(label="Response", lines=10),
    title="AI Agent with Web Access",
    description="This AI Agent has access to the internet. You can ask anything and it will search the web to get you your answer.",
    examples=[
        ["Differentiate between VectorDB and Vector Store"],
        ["What is RAG model?"],
        ["What is the current market cap of NVIDIA"]
    ]
)

In [18]:
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e266235a3f5a29a79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
